## 01 - Exploratory Data Analysis (EDA)
Fraude detection case - Artefact Intern Data Scientist Brazil

Goals of this step:
1. Understand the structure and data quality.
2. Understand the target (`flag_fraude`) in the historic 2025 database.
3. Find which variables separate fraud from non-fraud.
4. Check which columns are redundant or useless.
5. Check wheter January 2026 (the testing set) looks like the training data.

#### Configs:

In [2]:
from pathlib import Path
 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display,width", 150)

# Data paths
# 1. Training set
DATA_2025 = Path("src/data/transacoes_historico_2025.csv")
# 2. Testing set
DATA_2026 = Path("src\data\transacoes_jan2026.csv")
# 3. Resulting figures path
FIG_DIR = Path("outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Target definition
TARGET = "flag_fraude"

def save_fig(name: str) -> None:
    """Save the current figure so it can be reused in the presentation."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

<>:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\daviz\AppData\Local\Temp\ipykernel_19228\1170596327.py:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  DATA_2026 = Path("src\data\transacoes_jan2026.csv")


ModuleNotFoundError: No module named 'matplotlib'

#### Load Data

In [ ]:
df = pd.read_csv(DATA_2025, parse_dates=["datahora_transacao"])
df_jan =pd.read_csv(DATA_2026, parse_dates=["datahora_transacao"])

print("2025 (train):", df.shape)
print("Jan 2026 (test):", df_jan.shape)
df.head()

## 1. Structure and data quality
Basic quality checks

In [ ]:
print("Columns only in train :", set(df.columns) - set(df_jan.columns))
print("Columns only in Jan 26:", set(df_jan.columns) - set(df.columns))
print("\nMissing values (train):", int(df.isna().sum().sum()))
print("Duplicated rows (train):", int(df.duplicated().sum()))
print("Duplicated rows (Jan 26):", int(df_jan.duplicated().sum()))
print("\nTime coverage (train):", df.datahora_transacao.min(), "->", df.datahora_transacao.max())
print("Time coverage (Jan 26):", df_jan.datahora_transacao.min(), "->", df_jan.datahora_transacao.max())
df.dtypes


## 2. Target: class balance

In [ ]:
counts = df[TARGET].value_counts()
rate = df[TARGET].mean()
print(counts.to_string())
print(f"\nFraud rate: {rate:.2%}  (about 1 fraud every {1 / rate:.0f} transactions)")
print("Accuracy of a model that always predicts 'not fraud':", f"{1 - rate:.2%}")


#### Author's Inference
A trivial model that never flags fraud already reaches ~96% accuracy, so accuracy is a useless metric here. I will use precision, recall, F1 and PR-AUC, instead. 

#### Fraud rate over time

In [ ]:
monthly = df.groupby(df.datahora_transacao.dt.to_period("M")).agg(
    transactions=(TARGET, "size"), frauds=(TARGET, "sum"), fraud_rate=(TARGET, "mean")
)
print(monthly.round(4).to_string())
